In [2]:
import os
import random
import numpy as np
import cv2
import mediapipe as mp
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from PIL import Image

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [4]:
def crop_face(image):
    img = cv2.imread(image)

    mp_face_detection = mp.solutions.face_detection
    mp_drawing = mp.solutions.drawing_utils

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with mp_face_detection.FaceDetection(model_selection=5, min_detection_confidence=0.8) as face_detection:
        results = face_detection.process(img_rgb)

        if results.detections:
            for detection in results.detections:
                bboxC = detection.location_data.relative_bounding_box
                ih, iw, _ = img.shape

                x = max(int(bboxC.xmin * iw), 0)
                y = max(int(bboxC.ymin * ih) , 0)
                w = int(bboxC.width * iw)
                h = int(bboxC.height * ih)
                x2 = min(x + w, iw)
                y2 = min(y + h + 15, ih)

                face_crop = img[y:y2, x:x2]
                cv2.rectangle(img, (x, y), (x2, y2), (0, 255, 0), 2)
                
    return face_crop

In [5]:
class SiameseDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted(os.listdir(root_dir))
        self.image_paths = []

        for label in self.classes:
            class_path = os.path.join(root_dir, label)
            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)
                self.image_paths.append((img_path, label))

        self.label_to_images = {label: [] for label in self.classes}
        for path, label in self.image_paths:
            self.label_to_images[label].append(path)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        anchor_path, anchor_label = self.image_paths[idx]
        anchor_img = self.load_and_crop(anchor_path)

        # Positive image (cùng label)
        positive_candidates = self.label_to_images[anchor_label].copy()
        positive_candidates.remove(anchor_path)
        if not positive_candidates:
            positive_path = anchor_path
        else:
            positive_path = random.choice(positive_candidates)
        positive_img = self.load_and_crop(positive_path)

        # Negative image (khác label)
        negative_labels = [label for label in self.classes if label != anchor_label]
        negative_label = random.choice(negative_labels)
        negative_path = random.choice(self.label_to_images[negative_label])
        negative_img = self.load_and_crop(negative_path)

        if self.transform:
            anchor_img = self.transform(anchor_img)
            positive_img = self.transform(positive_img)
            negative_img = self.transform(negative_img)

        return anchor_img, positive_img, negative_img

    def load_and_crop(self, image_path):
        cropped = crop_face(image_path)
        if cropped is None:
            raise ValueError(f"Không đọc được ảnh: {image_path}")
        pil_image = Image.fromarray(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
        return pil_image


In [6]:
class SiameseDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted(os.listdir(root_dir))
        self.image_paths = []

        # lưu (path, label)
        for label in self.classes:
            class_path = os.path.join(root_dir, label)
            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)
                self.image_paths.append((img_path, label))

        # gom ảnh theo class để dễ chọn positive/negative
        self.label_to_images = {label: [] for label in self.classes}
        for path, label in self.image_paths:
            self.label_to_images[label].append(path)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # chọn anchor
        img1_path, img1_label = self.image_paths[idx]
        img1 = Image.open(img1_path).convert("RGB")

        # random: 50% tạo positive pair, 50% tạo negative pair
        if random.random() < 0.5:
            # Positive: cùng class
            img2_path = random.choice(self.label_to_images[img1_label])
            label = 0
        else:
            # Negative: khác class
            negative_label = random.choice([l for l in self.classes if l != img1_label])
            img2_path = random.choice(self.label_to_images[negative_label])
            label = 1

        img2 = Image.open(img2_path).convert("RGB")

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return (img1, img2), label

In [7]:
dataset = SiameseDataset(root_dir=r"C:\VSCode\Python\face_recognition\dataset\train",
                         transform=transform)

train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [8]:
class SiameseNet(nn.Module):
    def __init__(self, embedding_dim=128):
        super(SiameseNet, self).__init__()

        mobilenet = models.mobilenet_v3_small(weights="DEFAULT")

        self.feature_extractor = mobilenet.features  

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        last_channel = mobilenet.classifier[0].in_features
        self.fc = nn.Linear(last_channel, embedding_dim)

        self.dropout = nn.Dropout(0.5)

    def forward_once(self, x):
        x = self.feature_extractor(x)           
        x = self.avgpool(x)                     
        x = torch.flatten(x, 1)                
        x = self.dropout(F.relu(self.fc(x)))    
        x = F.normalize(x, p=2, dim=1)         
        return x

    def forward(self, x1, x2):
        out1 = self.forward_once(x1)
        out2 = self.forward_once(x2)
        return out1, out2


In [11]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        dist = F.pairwise_distance(output1, output2, p=2)

        loss_similar = (1 - label) * 0.5 * torch.pow(dist, 2)
        loss_dissimilar = label * 0.5 * torch.pow(torch.clamp(self.margin - dist, min=0.0), 2)

        loss = torch.mean(loss_similar + loss_dissimilar)
        return loss

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SiameseNet().to(device)
criterion = ContrastiveLoss(margin=1.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for (img1, img2), labels in train_loader:
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device).float()

        optimizer.zero_grad()
        out1, out2 = model(img1, img2)
        loss = criterion(out1, out2, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss = {avg_loss:.4f}")



Epoch 1/20, Loss = 0.2748
Epoch 2/20, Loss = 0.2496
Epoch 3/20, Loss = 0.2490
Epoch 4/20, Loss = 0.2466
Epoch 5/20, Loss = 0.2525
Epoch 6/20, Loss = 0.2486
Epoch 7/20, Loss = 0.2614
Epoch 8/20, Loss = 0.2626
Epoch 9/20, Loss = 0.2651
Epoch 10/20, Loss = 0.2706
Epoch 11/20, Loss = 0.2489
Epoch 12/20, Loss = 0.2770
Epoch 13/20, Loss = 0.2553
Epoch 14/20, Loss = 0.2334
Epoch 15/20, Loss = 0.2405
Epoch 16/20, Loss = 0.2718
Epoch 17/20, Loss = 0.2739
Epoch 18/20, Loss = 0.2266
Epoch 19/20, Loss = 0.2719
Epoch 20/20, Loss = 0.2625


In [ ]:
torch.save(model.state_dict(), r"C:\VSCode\Python\face_recognition\siamese_model.pth")
model.load_state_dict(torch.load(r"C:\VSCode\Python\face_recognition\siamese_model.pth"))
model.eval()    

SiameseNet(
  (feature_extractor): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=

In [18]:
from torchsummary import summary
summary(model, [(3, 224, 224), (3, 224, 224)], device=device.type)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 16, 112, 112]             432
       BatchNorm2d-2         [-1, 16, 112, 112]              32
         Hardswish-3         [-1, 16, 112, 112]               0
            Conv2d-4           [-1, 16, 56, 56]             144
       BatchNorm2d-5           [-1, 16, 56, 56]              32
              ReLU-6           [-1, 16, 56, 56]               0
 AdaptiveAvgPool2d-7             [-1, 16, 1, 1]               0
            Conv2d-8              [-1, 8, 1, 1]             136
              ReLU-9              [-1, 8, 1, 1]               0
           Conv2d-10             [-1, 16, 1, 1]             144
      Hardsigmoid-11             [-1, 16, 1, 1]               0
SqueezeExcitation-12           [-1, 16, 56, 56]               0
           Conv2d-13           [-1, 16, 56, 56]             256
      BatchNorm2d-14           [-1, 16,

In [19]:
reference_paths = []
for root, dirs, files in os.walk(r"C:\VSCode\Python\face_recognition\dataset\train"):
    count = 0
    for file in files:
        if file.endswith(".png") or file.endswith(".jpg"):
            reference_paths.append(os.path.join(root, file))
            count += 1
            if count >= 5:
                break

In [22]:
def predic_cosin_2_img(img_input1, img_input2):
    if isinstance(img_input1, np.ndarray):
        img1 = Image.fromarray(img_input1).convert('RGB')
    else:
        img1 = Image.open(img_input1).convert('RGB')

    img1 = transform(img1).unsqueeze(0).to(device)

    if isinstance(img_input2, np.ndarray):
        img2 = Image.fromarray(img_input2).convert('RGB')
    else:
        img2 = Image.open(img_input2).convert('RGB')

    img2 = transform(img2).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        embed_anchor, embed_target = model.forward(img1, img2)
        

    cosin = torch.nn.functional.cosine_similarity(embed_anchor, embed_target).item()

    return cosin

In [23]:
img1 = r"C:\VSCode\Python\face_recognition\screenshot_1754641209.png"
img2 = r"C:\VSCode\Python\face_recognition\screenshot_1754641221.png"

predic_cosin_2_img(img1, img2)

0.9970507621765137

In [ ]:
def predict_image(img_input, threshold=0.8):
    if isinstance(img_input, np.ndarray):
        img = Image.fromarray(img_input).convert('RGB')
    else:
        img = Image.open(img_input).convert('RGB')

    img2 = np.zeros((224, 224, 3), dtype=np.uint8)
    img2 = Image.fromarray(img2).convert('RGB')
    img1 = transform(img).unsqueeze(0).to(device)
    img2 = transform(img2).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        embed_anchor, embed_target = model.forward(img1, img2)

    distances = []

    count = 0
    dist = 0
    for ref_path in reference_paths:
        ref_img = Image.open(ref_path).convert('RGB')
        ref_img = transform(ref_img).unsqueeze(0).to(device)

        with torch.no_grad():
           embed_anchor, embed_ref = model.forward(ref_img)

        cosin = torch.nn.functional.cosine_similarity(embed_anchor, embed_ref).item()
        # dist += torch.nn.functional.pairwise_distance(embed_anchor, embed_ref).item()
        # count += 1
        # if count % 5 == 0:
        #     dist = dist / 5.0
        #     distances.append((ref_path, dist))
        #     count, dist = 0, 0

    distances.sort(key=lambda x: x[1])
    name = distances[0][0].split(os.sep)[-2]
    dist = distances[0][1]

    return name, dist

In [ ]:
predict_image(r"C:\VSCode\Python\face_recognition\dataset\test\chris_evans\chris_evans16.png")
